# B2-024-gpu-scientific-ml-capstone — Practice p12 — Solution

**Type:** constrained-coding · **Difficulty:** advanced · **Concepts:** open-ended-experiment-design

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import math

TOY_CONFIGS = {"lr=0.1": {"lr": 0.1}, "lr=0.2": {"lr": 0.2}, "lr=0.05": {"lr": 0.05}}

## Solution

`run_ablation` is a loop over configs (dict order) and seeds, with one fresh `StepBudget(steps_per_run)` per run.
After each run it checks `budget.used == steps_per_run`, so a run that stops early (or a `train_fn` that forgets to wrap an optimizer, which leaves `used` at 0) raises instead of silently getting a cheaper comparison; a run that tries to overspend is stopped by `StepBudget` itself.
The table holds the per-seed scores, their mean, and the **sample** standard deviation (divisor $r-1$, `torch.std`'s default); that is why at least two seeds are required.

In [ ]:
def run_ablation(configs, train_fn, seeds, steps_per_run):
    seeds = tuple(seeds)
    if len(seeds) < 2:
        raise ValueError("run_ablation needs at least two seeds for a sample standard deviation")
    table = {}
    for name, config in configs.items():
        scores = []
        for seed in seeds:
            budget = capstone_data.StepBudget(steps_per_run)
            score = float(train_fn(config, seed, budget))
            if budget.used != steps_per_run:
                raise RuntimeError(f"{name}, seed {seed}: used {budget.used} of {steps_per_run} steps")
            scores.append(score)
        r = len(scores)
        mean = sum(scores) / r
        std = math.sqrt(sum((s - mean) ** 2 for s in scores) / (r - 1))
        table[name] = {"scores": tuple(scores), "mean": mean, "std": std, "steps": steps_per_run * r}
    return table


def population_std(scores):
    m = sum(scores) / len(scores)
    return math.sqrt(sum((s - m) ** 2 for s in scores) / len(scores))


def toy_train_fn(config, seed, budget):
    w = torch.tensor(float(seed), requires_grad=True)
    optimizer = budget.wrap(torch.optim.SGD([w], lr=config["lr"]))
    while budget.remaining > 0:
        optimizer.zero_grad()
        loss = (w - 3.0) ** 2
        loss.backward()
        optimizer.step()
    return float((w.detach() - 3.0) ** 2)


def early_stop_train_fn(config, seed, budget):
    w = torch.tensor(float(seed), requires_grad=True)
    optimizer = budget.wrap(torch.optim.SGD([w], lr=config["lr"]))
    while budget.remaining > 1:                      # one step short
        optimizer.zero_grad()
        ((w - 3.0) ** 2).backward()
        optimizer.step()
    return float((w.detach() - 3.0) ** 2)


toy = run_ablation(TOY_CONFIGS, toy_train_fn, (0, 1, 2), 5)
for name, row in toy.items():
    print(name, row)

try:
    run_ablation(TOY_CONFIGS, early_stop_train_fn, (0, 1, 2), 5)
    early_raised = False
except RuntimeError as err:
    early_raised = True
    print("early stop ->", err)
try:
    run_ablation(TOY_CONFIGS, toy_train_fn, (0,), 5)
    one_seed_raised = False
except ValueError as err:
    one_seed_raised = True
    print("one seed ->", err)
pop_std = population_std(toy["lr=0.1"]["scores"])
print("population std for lr=0.1:", pop_std)

**A real ablation on the inverse task.**
The two configurations differ in exactly one factor, the input representation: an `8 → 32 → 32 → 3` MLP on standardized **log** magnitudes versus the same MLP on standardized **raw** magnitudes (Session 2's physics argument: the noise is multiplicative, so logs make it additive with constant scale).
Everything else is shared: width 32, full-batch Adam at learning rate $3\times10^{-3}$, 100 steps, seeds 0 and 1 (`torch.manual_seed(seed)` before building the network), statistics from the training rows only, score = mean **validation** position error. No test call is made.

In [ ]:
train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")


def inverse_train_fn(config, seed, budget):
    feats = (lambda m: torch.log(m)) if config["use_log"] else (lambda m: m)
    tr = feats(train_x)
    mu, sd = tr.mean(dim=0), tr.std(dim=0)
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 3))
    optimizer = budget.wrap(torch.optim.Adam(net.parameters(), lr=3e-3))
    z = (tr - mu) / sd
    for _ in range(config["steps"]):
        optimizer.zero_grad(set_to_none=True)
        loss = F.mse_loss(net(z), train_y)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        pred = net((feats(val_x) - mu) / sd)
    return capstone_data.position_error_per_example(pred, val_y).mean()


REAL_CONFIGS = {"log-32": {"use_log": True, "steps": 100}, "raw-32": {"use_log": False, "steps": 100}}
real = run_ablation(REAL_CONFIGS, inverse_train_fn, (0, 1), 100)
print(f"{'config':8s} {'seed 0':>8s} {'seed 1':>8s} {'mean':>8s} {'std':>8s} steps")
for name, row in real.items():
    print(f"{name:8s} {row['scores'][0]:8.4f} {row['scores'][1]:8.4f} {row['mean']:8.4f} {row['std']:8.4f} {row['steps']}")
paired = [a - b for a, b in zip(real["log-32"]["scores"], real["raw-32"]["scores"])]
print("per-seed differences log - raw:", [round(d, 4) for d in paired])

In [ ]:
mean_gap = real["log-32"]["mean"] - real["raw-32"]["mean"]
diff_std = math.sqrt(sum((d - sum(paired) / 2) ** 2 for d in paired) / (len(paired) - 1))
same_sign = all(d < 0 for d in paired) or all(d > 0 for d in paired)
demonstrated = same_sign and abs(mean_gap) > diff_std
print(f"mean gap {mean_gap:+.4f}, sample std of per-seed differences {diff_std:.4f}, same sign: {same_sign}, demonstrated: {demonstrated}")

**Is the difference demonstrated?**
No: at 100 steps the log configuration has the lower mean validation error (about $0.077$ versus $0.085$), but the two shared-seed differences have opposite signs (about $-0.025$ for seed 0 and $+0.010$ for seed 1), and the mean gap of about $0.008$ is smaller than the spread of the per-seed differences and of the raw configuration's own seed scores ($\hat\sigma\approx0.038$).
With two seeds and a 100-step budget, at which both networks are still far from converged, seed noise dominates, so the honest conclusion is "not demonstrated"; a decision would need more seeds or a longer equal budget per run (Session 2, §5).

### Answer check

In [ ]:
EXPECTED = {
    "lr=0.1": ((0.9663678407669067, 0.429496705532074, 0.10737410187721252), 0.5010795493920644, 0.4339477370769859),
    "lr=0.2": ((0.05441953241825104, 0.024186458438634872, 0.006046614609658718), 0.028217535155514877, 0.024437102507298938),
    "lr=0.05": ((3.138105869293213, 1.3947136402130127, 0.3486785590648651), 1.6271660228570302, 1.4091670456016205),
}
close = lambda a, b: math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
assert list(toy) == list(EXPECTED)
for name, (scores, mean, std) in EXPECTED.items():
    row = toy[name]
    assert isinstance(row["scores"], tuple) and all(close(a, b) for a, b in zip(row["scores"], scores))
    assert close(row["mean"], mean) and close(row["std"], std) and row["steps"] == 15
assert early_raised and one_seed_raised
assert close(pop_std, 0.3543168436246828) and not close(pop_std, toy["lr=0.1"]["std"])
assert list(real) == ["log-32", "raw-32"] and all(row["steps"] == 200 for row in real.values())
assert all(math.isfinite(s) for row in real.values() for s in row["scores"])
assert demonstrated is False          # matches the written conclusion